# Sesión 05 — Modelo estructural universal

El modelo se construye directamente desde `MI_EDIFICIO`, usando la biblioteca generada en la sesión 03.


In [1]:
from pathlib import Path
import sys, json, numpy as np
RAIZ=Path.cwd()
while not ((RAIZ/'temario.md').is_file() and (RAIZ/'PUSHOVER_PORTABLE').is_dir()): RAIZ=RAIZ.parent
PORTABLE=RAIZ/'PUSHOVER_PORTABLE'; sys.path.insert(0,str(PORTABLE))
from ejecutar_pushover import cargar_entrada_maestra
from sesion_03.biblioteca_rotulas import leer_contrato_sesion03
from sesion_05.contrato_portable import construir_contrato_sesion05, guardar_contrato_sesion05
from sesion_05.modelo_portico import ensamblar_rigidez_global
RUTA_MAESTRA=PORTABLE/'casos'/'edificio_6pisos'/'entrada_maestra.json'
entrada,_,_=cargar_entrada_maestra(RUTA_MAESTRA); CASO_ID=entrada['caso_id']
base=RAIZ/'resultados_curso_portable'/CASO_ID
biblioteca=leer_contrato_sesion03(base/'sesion_03'/'contrato_sesion03.json')['biblioteca_rotulas']
contrato=construir_contrato_sesion05(entrada,biblioteca)
nodos={int(k):np.asarray(v,dtype=float) for k,v in contrato['nodos'].items()}
for e in contrato['elementos']:
    L=np.linalg.norm(nodos[int(e['j'])]-nodos[int(e['i'])])-e['brazo_i_cm']-e['brazo_j_cm']
    assert L>0
K,_,_=ensamblar_rigidez_global(nodos,contrato['elementos'])
restr=np.asarray(contrato['restricciones_gdl'],dtype=int); libres=np.setdiff1d(np.arange(K.shape[0]),restr)
Kff=K[np.ix_(libres,libres)]
assert np.allclose(K,K.T) and np.linalg.matrix_rank(Kff)==len(libres)
salida=base/'sesion_05'; salida.mkdir(parents=True,exist_ok=True)
ruta=salida/'contrato_sesion05.json'; guardar_contrato_sesion05(contrato,ruta)
print('Caso:',CASO_ID,'| nodos:',len(nodos),'| elementos:',len(contrato['elementos']),'| rótulas:',len(contrato['rotulas']))
print('Contrato:',ruta)


Caso: edificio_6pisos | nodos: 84 | elementos: 120 | rótulas: 240
Contrato: d:\DOCENCIA\PRINBEL\PUSHOVER PORTICOS\resultados_curso_portable\edificio_6pisos\sesion_05\contrato_sesion05.json
